In [1]:
import torch
import triton

In [2]:
!pip install triton

In [3]:
print(triton.__version__)

3.6.0


In [4]:
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
  pid=tl.program_id(0)
  ceils=tl.ceil((n_elements/BLOCK_SIZE))
  block_start=pid*BLOCK_SIZE
  offsets=block_start+tl.arange(0, BLOCK_SIZE)
  mask=offsets<n_elements

  x=tl.load(x_ptr+offsets,mask=mask)
  y=tl.load(y_ptr+offsets, mask=mask)

  output=x+y
  tl.store(out_ptr+offsets, output, mask=mask)




In [5]:
x = torch.randn(10000, device="cuda")
y = torch.randn(10000, device="cuda")
out = torch.empty_like(x)

In [6]:
BLOCK_SIZE = 1024

In [7]:
grid = (triton.cdiv(x.numel(), BLOCK_SIZE),)

In [9]:
add_kernel[grid](
    x,
    y,
    out,
    x.numel(),
    BLOCK_SIZE=1024,
)


In [10]:
out=x+y

In [37]:
import time
start = time.perf_counter()

for _ in range(5000):
    out=x+y

torch.cuda.synchronize()
end = time.perf_counter()

print((end - start) / 5000)

1.3134623599989936e-05


In [38]:
import time

torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(5000):
    add_kernel[grid](
        x,
        y,
        out,
        x.numel(),
        BLOCK_SIZE=1024,
    )

torch.cuda.synchronize()
end = time.perf_counter()

print("Triton:", (end - start) / 5000)

Triton: 1.4598350599999321e-05
